In [ ]:
#encoder
import torch.nn as nn
conv2= nn.Conv2d(3,32,4,stride=2,padding=1)
conv22=nn.Conv2d(32,64,4,stride=2,padding=1)
conv222=nn.Conv2d(64,128,4,stride=2,padding=1)
x=conv2(train_feature)
x=conv22(x)
x=conv222(x)
print(x.shape)

torch.Size([64, 128, 8, 8])


In [ ]:
#flattening and appplying linear nn
import torch
latent_dimension=128
flatten_layer=nn.Flatten()
in_features = 128 * 8 * 8
latent_dim = 128
fc_mu = nn.Linear(in_features, latent_dim)
fc_logvar = nn.Linear(in_features, latent_dim)
x_flat = flatten_layer(x)
mu = fc_mu(x_flat)
logvar = fc_logvar(x_flat)

In [ ]:
#reparametirzation\
def reparameterize(mu, logvar):

    std = torch.exp(0.5 * logvar)
    eps = torch.randn_like(std)
    z=mu+std*eps
    return z

In [ ]:
#decoder
class Decoder(nn.Module):
    def __init__(self, latent_dim=128):
        super(Decoder, self).__init__()

        # 1. Linear layer jo 128 numbers ko wapas flat chataai (8192 numbers) banayegi
        self.fc_init = nn.Linear(latent_dim, 128 * 8 * 8)

        # 2. Teeno layers ko alag-alag variables mein define kiya (Tera waala style)
        self.m1 = nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1)
        self.m2 = nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1)
        self.m3 = nn.ConvTranspose2d(32, 3, kernel_size=4, stride=2, padding=1)

        # Activations ko bhi alag se rakh lete hain
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, z):
        # Step 1: z ko flat linear output mein badla -> Shape: [64, 8192]
        x = self.fc_init(z)

        # Step 2: Wapas 3D dabba banaya -> Shape: [64, 128, 8, 8]
        x = x.view(-1, 128, 8, 8)

        # Step 3: Pehli layer se guzara aur phulaya -> Shape: [64, 64, 16, 16]
        x = self.m1(x)
        x = self.relu(x)

        # Step 4: Dusri layer se guzara aur phulaya -> Shape: [64, 32, 32, 32]
        x = self.m2(x)
        x = self.relu(x)

        # Step 5: Aakhiri layer se guzara (No ReLU, directly Sigmoid) -> Shape: [64, 3, 64, 64]
        x = self.m3(x)
        reconstructed_img = self.sigmoid(x)

        return reconstructed_img
decoder = Decoder(latent_dim=128)
print("Decoder ekdum ready hai!")

Decoder ekdum tere style mein ready hai!
